# `lina` vs `lina_cpp` - parity & performance comparison

This notebook does two things:

1. **Parity** - For every function that has a C++ implementation, run the
   same inputs through both backends and report the maximum elementwise
   numerical difference.
2. **Performance** - Time each function in both backends across a range
   of array sizes and report the speedup.

Both packages are imported side-by-side, so user code that already does
`import lina` keeps working unchanged. Functions that haven't been ported
yet (high-level llowfsc loop, telem helpers, hardware control) are simply
re-exported from `lina` by `lina_cpp`, so calling them through either
package is identical.

Prereq: install pure-python `lina` from `lina_kian`, then install
`lina_cpp` from this repo:

`python -m pip install -e /home/adams/lina_test/lina_kian`

`python -m pip install -e /home/adams/lina_test/lina_cpp/lina_cpp`

In [ ]:
import time
import numpy as np
import scipy

# Set up the backend so lina.utils.* uses numpy (not cupy by default).
import lina
from lina import math_module
math_module.update_np(np)
math_module.update_scipy(scipy)

import lina_cpp
print(f'lina      version: {lina.__version__}')
print(f'lina_cpp  version: {lina_cpp.__version__}')
print(f'cfitsio enabled in lina_cpp: {lina_cpp.utils.fits_available()}')

## Helpers

In [ ]:
def time_call(fn, *args, repeat=5, **kwargs):
    '''Best-of-N microbenchmark. Returns (median_ms, result).'''
    fn(*args, **kwargs)  # warm up (JIT, FFTW plan caching, etc.)
    samples = []
    out = None
    for _ in range(repeat):
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        samples.append(time.perf_counter() - t0)
    return float(np.median(samples) * 1e3), out

def report(name, py_ms, cpp_ms, max_diff):
    speedup = py_ms / cpp_ms if cpp_ms > 0 else float('inf')
    print(f'  {name:32s}  python={py_ms:7.2f} ms  c++={cpp_ms:7.2f} ms  '
          f'speedup={speedup:5.2f}x  max-diff={max_diff:.2e}')

## FFT / IFFT

In [ ]:
for N in [64, 128, 256, 512, 1024]:
    print(f'\nN = {N}x{N}')
    x = np.random.RandomState(0).standard_normal((N, N)).astype(np.complex128)
    py_ms, py_out  = time_call(lambda a: np.asarray(lina.props.fft(a)), x)
    cpp_ms, cpp_out = time_call(lina_cpp.props.fft, x)
    report('fft',  py_ms, cpp_ms, np.max(np.abs(py_out - cpp_out)))
    py_ms, py_out  = time_call(lambda a: np.asarray(lina.props.ifft(a)), x)
    cpp_ms, cpp_out = time_call(lina_cpp.props.ifft, x)
    report('ifft', py_ms, cpp_ms, np.max(np.abs(py_out - cpp_out)))

## Matrix Fourier transform

In [ ]:
for N, npsf in [(128, 64), (256, 128), (512, 256)]:
    print(f'\nN = {N}x{N}, npsf = {npsf}x{npsf}')
    pupil = np.random.RandomState(1).standard_normal((N, N)).astype(np.complex128)

    py_ms,  py_out  = time_call(lambda p: np.asarray(lina.props.mft_forward(p, N, npsf, 0.5)), pupil)
    cpp_ms, cpp_out = time_call(lambda p: lina_cpp.props.mft_forward(p, N, npsf, 0.5), pupil)
    report('mft_forward', py_ms, cpp_ms, np.max(np.abs(py_out - cpp_out)))

    fpwf = py_out
    py_ms,  py_out  = time_call(lambda f: np.asarray(lina.props.mft_reverse(f, 0.5, N, N)), fpwf)
    cpp_ms, cpp_out = time_call(lambda f: lina_cpp.props.mft_reverse(f, 0.5, N, N), fpwf)
    report('mft_reverse', py_ms, cpp_ms, np.max(np.abs(py_out - cpp_out)))

## Wave propagation: angular spectrum + Fresnel TF

In [ ]:
for N in [128, 256, 512]:
    print(f'\nN = {N}x{N}')
    wf = np.random.RandomState(2).standard_normal((N, N)).astype(np.complex128)

    py_ms,  py_out  = time_call(lambda a: np.asarray(lina.props.ang_spec(a, 633e-9, 0.5, 1e-5)), wf)
    cpp_ms, cpp_out = time_call(lambda a: lina_cpp.props.ang_spec(a, 633e-9, 0.5, 1e-5), wf)
    report('ang_spec',     py_ms, cpp_ms, np.max(np.abs(py_out - cpp_out)))

    py_ms,  py_out  = time_call(lambda: np.asarray(lina.props.get_fresnel_TF(1e-3, N, 633e-9, 20.0)))
    cpp_ms, cpp_out = time_call(lambda: lina_cpp.props.get_fresnel_TF(1e-3, N, 633e-9, 20.0))
    report('get_fresnel_TF', py_ms, cpp_ms, np.max(np.abs(py_out - cpp_out)))

## Mask construction

In [ ]:
for N in [256, 512, 1024]:
    print(f'\nN = {N}')
    py_ms, py_out = time_call(lambda: np.asarray(lina.utils.create_annular_mask(N, 1.0/N, 0.05, 0.45)))
    cpp_ms, cpp_out = time_call(lambda: lina_cpp.utils.create_annular_mask(N, 1.0/N, 0.05, 0.45))
    report('create_annular_mask', py_ms, cpp_ms,
           np.max(np.abs(py_out.astype(float) - cpp_out.astype(float))))

## FITS I/O

Round-trip a 1024x1024 array via FITS (lina uses astropy, lina_cpp uses cfitsio directly).

In [ ]:
import tempfile, os
from astropy.io import fits as apfits

for N in [256, 1024]:
    print(f'\nN = {N}x{N}')
    a = np.random.RandomState(7).standard_normal((N, N))
    with tempfile.TemporaryDirectory() as td:
        # save
        py_ms, _ = time_call(lambda: lina.utils.save_fits(os.path.join(td, 'py.fits'),  a, ow=True, quiet=True))
        cpp_ms, _ = time_call(lambda: lina_cpp.utils.save_fits(os.path.join(td, 'cpp.fits'), a, ow=True, quiet=True))
        report('save_fits', py_ms, cpp_ms, 0.0)
        # load
        py_ms, py_out = time_call(lambda: lina.utils.load_fits(os.path.join(td, 'py.fits')))
        cpp_ms, cpp_out = time_call(lambda: lina_cpp.utils.load_fits(os.path.join(td, 'cpp.fits')))
        report('load_fits', py_ms, cpp_ms, np.max(np.abs(np.asarray(py_out) - cpp_out)))

## Drop-in replacement check

Demonstrates that the entire `lina_cpp` package is API-compatible with `lina`. A user can swap a single import line and the rest of the code is unchanged.

In [ ]:
# Original code:
#   import lina
#   mask = lina.utils.create_annular_mask(256, 1/256, 0.05, 0.45)
#   fft = lina.props.fft(arr)
#
# New code (one line changed):
import lina_cpp as lina_swap
lina_swap.math_module.update_np(np)
lina_swap.math_module.update_scipy(scipy)

mask = lina_swap.utils.create_annular_mask(256, 1/256, 0.05, 0.45)
arr  = np.random.RandomState(0).standard_normal((256, 256)).astype(np.complex128)
fft  = lina_swap.props.fft(arr)
print('shapes:', mask.shape, fft.shape)